# 📓 Notebook 1｜資料與特徵向量：重現課本 Fig 1.2

> 對應講義 **Part 2**（知識地圖站 2）
>
> 課本故事：良性病灶（class A）與惡性病灶（class B）的影像資料庫，每張圖量測「平均亮度 mean」與「標準差 std」兩個特徵 → 畫成散點圖 → 一條直線就能分開。
>
> 我們把故事「倒著演」：**先製造出特性不同的兩類樣本**，再畫出課本那張圖。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)   # 固定種子 → 每次執行的圖都一樣

def gauss(n, mean, std):
    return mean + std * rng.standard_normal(n)

# class A：良性 —— 平均亮度較低且集中（亮暗差異小）
mean_A = gauss(25, 3.0, 1.4)      # 25 張圖的 mean 特徵
std_A  = gauss(25, 3.0, 1.4)      # 25 張圖的 std 特徵
# class B：惡性 —— 平均亮度較高、且樣本間差異大（std 大）
mean_B = gauss(25, 7.0, 1.4)
std_B  = gauss(25, 6.0, 1.4)

XA = np.stack([mean_A, std_A], axis=1)   # 特徵矩陣：25 個樣本 × 2 維
XB = np.stack([mean_B, std_B], axis=1)
print('class A 特徵矩陣 X_A:', XA.shape, ' class B 特徵矩陣 X_B:', XB.shape)

In [ ]:
# 畫出課本 Fig 1.2 的散點圖（含「未知樣本」星號）
plt.figure(figsize=(6, 5))
plt.scatter(XA[:, 0], XA[:, 1], marker='s', s=50, c='#2563eb', label='class A（良性）')
plt.scatter(XB[:, 0], XB[:, 1], marker='+', s=60, c='#dc2626', label='class B（惡性）')
# 一個「未知」的新樣本（人工捏造：看起來比較像 A）
plt.scatter([4.3], [3.2], marker='*', s=260, c='#f59e0b', label='未知樣本 ∗')
# 手畫一條決策線（很粗略，nb2 會「自動找最佳線」）
plt.plot([2, 9], [7.6, 1.6], 'k-', lw=1.5, label='決策線（目測）')
plt.xlim(0, 10); plt.ylim(0, 10)
plt.xlabel('特徵 x1 = 平均亮度 mean'); plt.ylabel('特徵 x2 = 亮度標準差 std')
plt.legend(); plt.grid(alpha=0.3); plt.show()

### ✏️ 練習 1
把 `rng` 的種子從 `42` 改為其他數字重新執行——形狀會變，但「兩群分得開」的本質不變。為什麼？（提示：兩類的**產生機制**不同 → 這就是「特徵是隨機變數」的意思）

### ✏️ 練習 2（關鍵！）
觀察 A、B 在「哪個特徵」上最不重疊？試試把 x 軸換成 std、y 軸換成 mean 畫一張圖，看看分離度是否一樣？── 這就是「特徵選擇」問題的起點（課本四大問題之二）。

## 把「標籤」加進來

監督式學習需要標籤：把兩類併成一份資料集，`y=0` 表良性、`y=1` 表惡性。

In [ ]:
X = np.vstack([XA, XB])     # 50 個樣本 × 2 維
y = np.array([0] * 25 + [1] * 25)
print('資料集 X:', X.shape, ' 標籤 y:', y.shape)
print('前 3 個訓練樣本（含標籤）:')
for i in range(3):
    print(f'  x = {X[i].round(3)}  →  y = {y[i]} ({"良性" if y[i]==0 else "惡性"})')

## 🏆 小結：你在 nb1 學會了
1. **資料的長相**：特徵矩陣 $X$（樣本 × 特徵）＋ 標籤 $y$
2. **特徵向量**：$\mathbf{x}=[x_1,x_2]^T$ 唯一對應一個樣本
3. **可視分離性**：好的特徵讓兩類在特徵空間分得開（課本 Fig 1.2 的由來）
4. **隨機性**：改種子圖會變，但機制不變——後續所有章節都在「對抗這個隨機性」